## Feature importance with Random Forest

In [1]:
from nbformat.sign import algorithms
# we can check their relative importance with build in funcs
# for this one ima use iris datset
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

iris = load_iris()
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
rf = RandomForestClassifier()
rf.fit(X_train, y_train)

for name, score in zip(iris['feature_names'], rf.feature_importances_):
    print(name, score)

# 80% of the weight is on petal length and width
# random forest very handy for feature selection!!!

sepal length (cm) 0.10038643106849292
sepal width (cm) 0.027009732937659476
petal length (cm) 0.4468533588105895
petal width (cm) 0.42575047718325804


## AdaBoost (Adaptive Boosting)

In [3]:
# boosting by definition is training models sequentially, its like train weak model and next model gonna learn from the mistakes of the previous model
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

ada_clf = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=5),
    n_estimators=200,
    learning_rate=0.5,
    random_state=42,
#   algorithm='SAMME.R', in the book there is this parameter but here it says adabooster doesnt have this parameter
)
ada_clf.fit(X_train, y_train)

,"estimator estimator: object, default=NoneThe base estimator from which the boosted ensemble is built.Support for sample weighting is required, as well as proper``classes_`` and ``n_classes_`` attributes. If ``None``, thenthe base estimator is :class:`~sklearn.tree.DecisionTreeClassifier`initialized with `max_depth=1`... versionadded:: 1.2 `base_estimator` was renamed to `estimator`.",DecisionTreeC...r(max_depth=5)
,"n_estimators n_estimators: int, default=50The maximum number of estimators at which boosting is terminated.In case of perfect fit, the learning procedure is stopped early.Values must be in the range `[1, inf)`.",200
,"learning_rate learning_rate: float, default=1.0Weight applied to each classifier at each boosting iteration. A higherlearning rate increases the contribution of each classifier. There isa trade-off between the `learning_rate` and `n_estimators` parameters.Values must be in the range `(0.0, inf)`.",0.5
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given at each `estimator` at eachboosting iteration.Thus, it is only used when `estimator` exposes a `random_state`.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels.","ndarray[int64](3,)","[0,1,2]"
estimator_ estimator_: estimatorThe base estimator from which the ensemble is grown... versionadded:: 1.2 `base_estimator_` was renamed to `estimator_`.,DecisionTreeClassifier,DecisionTreeC...r(max_depth=5)
estimator_errors_ estimator_errors_: ndarray of floatsClassification error for each estimator in the boostedensemble.,"ndarray[float64](200,)","[0.01,0.01,0. ,...,1. ,1. ,1. ]"
estimator_weights_ estimator_weights_: ndarray of floatsWeights for each estimator in the boosted ensemble.,"ndarray[float64](200,)","[2.74,2.79,1. ,...,0. ,0. ,0. ]"
estimators_ estimators_: list of classifiersThe collection of fitted sub-estimators.,list,"[DecisionTreeC...te=1608637542), DecisionTreeC...te=1273642419), DecisionTreeC...te=1935803228)]"
"feature_importances_ feature_importances_: ndarray of shape (n_features,)The impurity-based feature importances if supported by the``estimator`` (when based on decision trees).Warning: impurity-based feature importances can be misleading forhigh cardinality features (many unique values). See:func:`sklearn.inspection.permutation_importance` as an alternative.","ndarray[float64](4,)","[0.01,0.02,0.69,0.27]"


## Gradient Boosting

In [ ]:
# Gradient boosting also train models sequentially but he pays more attention to residuals

# when we calculate by hand we train one model and after second model with X and y2 = y - model1.predict(X), this is how it calculates on the background
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score
import numpy as np

gb_clf = GradientBoostingClassifier(n_estimators=200, learning_rate=0.5, max_depth=5, random_state=42)
gb_clf.fit(X_train, y_train)

errors = [accuracy_score(y_test, y_pred) for (y_pred) in gb_clf.predict(X_test)]

bst_n_estimators = np.argmin(errors)

gbrt_best = GradientBoostingClassifier(max_depth=5, n_estimators=bst_n_estimators, learning_rate=0.5, random_state=42)
gbrt_best.fit(X_train, y_train)
print(gbrt_best.score(X_test, y_test))

# cant run because dataset dont match with that the book intended to do and book doesnt tell what dataset to use

### Stopping training early if validation error doesn't improve next model

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
gbrt = GradientBoostingRegressor(max_depth=2, warm_start=True, random_state=42)

min_val_error = float('inf')
error_going_up = 0
for n_estimators in range(1, 120):
    gbrt.n_estimators = n_estimators
    gbrt.fit(X_train, y_train)
    y_pred = gbrt.predict(X_test)
    val_error = mean_squared_error(y_test, y_pred)
    if val_error < min_val_error:
        min_val_error = val_error
        error_going_up = 0
    else:
        error_going_up += 1
        if error_going_up == 5:
            break # 5 times error going up

## Exercises on page 213

In [ ]:
# 1. If you have trained five different models on the exact same training data, and
# they all achieve 95% precision, is there any chance that you can combine these
# models to get better results? If so, how? If not, why?
Answer1 = ('Yes, it is possible to combine the five models to achieve better results. Even though all models have the same precision (95%), they might make mistakes on different instances '
           'By combining their predictions using an ensemble method, such as a voting classifier, we can potentially achieve better performance than individual models')

# 2. What is the difference between hard and soft voting classifiers?
Answer2 = ('Hard voting: Combines predictions by selecting the class that receives the majority of votes from individual classifiers '
           'Soft voting: Combines predictions by calculating the average predicted probabilities for each class and selecting the class with the highest average probability')
# in short hard vote one final prediction, soft vote calculate mean of probabilities

# 3. Is it possible to speed up training of a bagging ensemble by distributing it across
# multiple servers? What about pasting ensembles, boosting ensembles, random
# forests, or stacking ensembles?
Answer3 = ('Training bagging, pasting, and Random Forest ensembles can be distributed across multiple servers because their models can be trained independently in parallel '
           'Boosting ensembles are more difficult to distribute because each model depends on the predictions and errors of previously trained models '
           'Stacking ensembles can also use parallel training for their base models, but the final blender must wait until their predictions are generated')

# 4. What is the benefit of out-of-bag evaluation?
Answer4 = ('Out-of-bag evaluation allows us to estimate the performance of a bagging ensemble without requiring a separate validation set. Since each model is trained on a random sample of the training data, some instances are left out. These unseen instances are used to evaluate the models and estimate their overall performance')


# 5. What makes Extra-Trees more random than regular Random Forests? How can
# this extra randomness help? Are Extra-Trees slower or faster than regular Random Forests?
Answer5 = ('Extra-Trees are more random than Random Forests because instead of searching for the best possible threshold for each feature, they choose random thresholds '
           'This extra randomness can reduce variance and help prevent overfitting, although it can increase bias '
           'Extra-Trees are usually faster to train because they do not need to search for the best thresholds')


# 6. If your AdaBoost ensemble underfits the training data, what hyperparameters
# should you tweak and how?
Answer6 = ('If an AdaBoost ensemble underfits the training data, we can increase the number of estimators so that more weak learners can be added '
           'We can also increase the learning rate to give more importance to each new learner and help the model learn more complex patterns')


# 7. If your Gradient Boosting ensemble overfits the training set, should you increase
# or decrease the learning rate?
Answer7 = ('We should decrease the learning rate because it makes each tree contribute less to the final prediction and can help reduce overfitting '
           'However, we may need to increase the number of estimators to compensate for the lower learning rate')

### Question 8

In [9]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

mnist = fetch_openml('mnist_784', version=1, as_frame=False)

X = mnist.data.astype('float32') / 255.0
y = mnist.target.astype('int8')

# Split into training, validation and test sets
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=10000, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=10000, random_state=42, stratify=y_train_val
)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(50000, 784)
(10000, 784)
(10000, 784)


In [10]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# Random Forest
rf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)

# Extra Trees
et = ExtraTreesClassifier(n_estimators=100, n_jobs=-1, random_state=42)
et.fit(X_train, y_train)

# SVM
svm = SVC(kernel='rbf', C=5, gamma='scale')
svm.fit(X_train, y_train)

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",5
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide <shrinking_svm>`.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide <scores_probabilities>`...deprecated:: 1.9 The `probability` parameter is deprecated and will be removed in 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`.",'deprecated'
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [11]:
rf_pred = rf.predict(X_val)
et_pred = et.predict(X_val)
svm_pred = svm.predict(X_val)

print("Random Forest:", accuracy_score(y_val, rf_pred))
print("Extra Trees:", accuracy_score(y_val, et_pred))
print("SVM:", accuracy_score(y_val, svm_pred))

Random Forest: 0.9703
Extra Trees: 0.9735
SVM: 0.9852


In [12]:
from sklearn.ensemble import VotingClassifier

voting_clf = VotingClassifier(
    estimators=[
        ('rf', rf),
        ('et', et),
        ('svm', svm)
    ],
    voting='hard'
)

voting_clf.fit(X_train, y_train)

voting_pred = voting_clf.predict(X_val)

print("Voting Classifier:", accuracy_score(y_val, voting_pred))

Voting Classifier: 0.9768


In [13]:
test_pred = voting_clf.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, test_pred))

Test Accuracy: 0.974
